In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy functions.py
COPY functions.py .

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm==4.64.1
numpy==1.23.4
pandas==1.2.4
boto3==1.24.59
matplotlib==3.7.1

Writing requirements.txt


### Write ```functions.py```

In [4]:
%%writefile functions.py

# functions
import pandas as pd
import numpy as np
import json
from pprint import pprint
from tqdm import tqdm
import matplotlib.pyplot as plt
import boto3

# get df info
def get_df_info(df, str_datecol, str_dirname_output, str_filename='dict_df_info.json'):
    # nrows/ncols
    int_nrows, int_ncols = df.shape
    # total obs
    int_obs_total = int_nrows * int_ncols
    # tot na
    int_n_missing = np.sum(df.isnull().sum())
    # create dict
    dict_df_info = {
        'int_nrows': int_nrows,
        'int_ncols': int_ncols,
        'int_obs_total': int_obs_total,
        'date_min': str(np.min(df[str_datecol])),
        'date_max': str(np.max(df[str_datecol])),
        'flt_mean_target': np.mean(df['target']),
        'flt_propna': int_n_missing / int_obs_total,
    }
    # write to .json
    json.dump(dict_df_info, open(f'{str_dirname_output}/{str_filename}', 'w'))
    # show
    #pprint(dict_df_info)

# get descriptives for each column
def get_descriptives_by_column(df, str_dirname_output, str_filename='df_descriptives.csv'):
    # get descriptives
    list_dict_row = []
    for col in tqdm (df.columns):
        # save as series
        ser_col = df[col]
        # get proportion nan
        flt_prop_na = ser_col.isnull().mean()
        if flt_prop_na == 1.0:
            # create row
            dict_row = {
                'feature': col,
                'dtype': np.nan,
                'propna': flt_prop_na,
                'min': np.nan,
                'max': np.nan,
                'range': np.nan,
                'std': np.nan,
                'mean': np.nan,
                'median': np.nan,
                'mode': np.nan,
                'n_unique': 0,
                'prop_unique': 0,
                'prop_negative': np.nan,
                'prop_min': np.nan,
                'prop_max': np.nan,
                'prop_zero': np.nan,
            }
            # append
            list_dict_row.append(dict_row)
            # skip the rest of the iteration
            continue
        # get data type
        str_dtype = ser_col.dtype
        # if value
        if str_dtype in ['float64', 'int64']:
            val_min, val_max, val_mean, val_std, val_median = ser_col.min(), ser_col.max(), ser_col.mean(), ser_col.std(), ser_col.median()
            val_range = val_max - val_min
            val_mode, int_n_unique = ser_col.mode().iloc[0], ser_col.nunique()
            flt_prop_unique = int_n_unique / len(ser_col.dropna())
            flt_prop_negative = len(ser_col[ser_col<0]) / len(ser_col.dropna())
            flt_prop_min = len(ser_col[ser_col==val_min]) / len(ser_col.dropna())
            flt_prop_max = len(ser_col[ser_col==val_max]) / len(ser_col.dropna())
            flt_prop_zero = len(ser_col[ser_col==0]) / len(ser_col.dropna())
        # if object
        if str_dtype == 'O':
            val_min, val_max, val_std, val_mean, val_median = np.nan, np.nan, np.nan, np.nan, np.nan
            val_range = np.nan
            val_mode, int_n_unique = ser_col.mode().iloc[0], ser_col.nunique()
            flt_prop_unique = int_n_unique / len(ser_col.dropna())
            flt_prop_negative = np.nan 
            flt_prop_min = np.nan
            flt_prop_max = np.nan
            flt_prop_zero = np.nan
        # if dtm
        if str_dtype == 'datetime64[ns]':
            val_min, val_max, val_mean, val_std, val_median = ser_col.min(), ser_col.max(), ser_col.mean(), np.nan, np.nan
            val_range = val_max - val_min
            val_mode, int_n_unique = ser_col.mode().iloc[0], ser_col.nunique()
            flt_prop_unique = int_n_unique / len(ser_col.dropna())
            flt_prop_negative = np.nan 
            flt_prop_min = len(ser_col[ser_col==val_min]) / len(ser_col.dropna())
            flt_prop_max = len(ser_col[ser_col==val_max]) / len(ser_col.dropna())
            flt_prop_zero = np.nan
        # create row
        dict_row = {
            'feature': col,
            'dtype': str_dtype,
            'propna': flt_prop_na,
            'min': val_min,
            'max': val_max,
            'range': val_range,
            'std': val_std,
            'mean': val_mean,
            'median': val_median,
            'mode': val_mode,
            'n_unique': int_n_unique,
            'prop_unique': flt_prop_unique,
            'prop_negative': flt_prop_negative,
            'prop_min': flt_prop_min,
            'prop_max': flt_prop_max,
            'prop_zero': flt_prop_zero,
        }
        # append
        list_dict_row.append(dict_row)
    # make df
    df_descriptives = pd.DataFrame(list_dict_row)
    # order cols
    df_descriptives.columns = [
        'feature',
        'dtype',
        'propna',
        'min',
        'max',
        'range',
        'std',
        'mean',
        'median',
        'mode',
        'n_unique',
        'prop_unique',
        'prop_negative',
        'prop_min',
        'prop_max',
        'prop_zero',
    ]
    df_descriptives.sort_values(by='propna', ascending=False, inplace=True)
    df_descriptives.to_csv(f'{str_dirname_output}/{str_filename}', index=False)
    # return
    return df_descriptives

# plot proportion NaN overall
def plot_proportion_nan(df, str_dirname_output, str_filename='plt_prop_nan.png'):
    # get int_n_missing
    int_n_missing = np.sum(df.isnull().sum())
    # get int_obs_total
    int_obs_total = df.shape[0] * df.shape[1]
    # create axis
    fig, ax = plt.subplots(figsize=(9,5))
    # title
    ax.set_title('Pie Chart of Missing Values')
    ax.pie(
        x=[int_n_missing, int_obs_total], 
        colors=['y', 'c'],
        explode=(0, 0.1),
        labels=['Missing', 'Non-Missing'], 
        autopct='%1.1f%%',
    )
    # save fig
    plt.savefig(f'{str_dirname_output}/{str_filename}', bbox_inches='tight')
    # close
    plt.close()

# plot data type frequency
def plot_data_type_frequency(df, str_dirname_output, str_filename='plt_dtype.png'):
    # get numeric
    list_cols_numeric = []
    for col in tqdm (df.columns):
        if df[col].dtype in ['int64', 'float64']:
            list_cols_numeric.append(col)
    # get non-numeric
    list_cols_non_numeric = [col for col in df.columns if col not in list_cols_numeric]
    # get number of columns
    int_ncols = df.shape[1]
    # % numeric
    flt_pct_numeric = (len(list_cols_numeric) / int_ncols) * 100
    # % non-numeric
    flt_pct_non_numeric = (len(list_cols_non_numeric) / int_ncols) * 100
    # create ax
    fig, ax = plt.subplots(figsize=(9,5))
    # title
    ax.set_title(f'{flt_pct_numeric:0.4}% Numeric, {flt_pct_non_numeric:0.4}% Non-Numeric (N = {int_ncols})')
    # y label
    ax.set_ylabel('Frequency')
    # bar plot
    ax.bar(['Numeric','Non-Numeric'], [len(list_cols_numeric), len(list_cols_non_numeric)])
    # save plot
    plt.savefig(f'{str_dirname_output}/{str_filename}', bbox_inches='tight')
    # close
    plt.close()

# plot target
def plot_target(ser_target, str_dirname_output, str_filename='plt_target.png'):
    # get the total positive
    int_tot_pos = np.sum(ser_target)
    # get total
    int_total = len(ser_target)
    # get the toeal negative
    int_tot_neg = int_total - int_tot_pos
    # get pct negative class
    flt_pct_negative = (int_tot_neg / int_total) * 100
    # get pct positive class
    flt_pct_positive = (int_tot_pos / int_total) * 100
    # create axis
    fig, ax = plt.subplots(figsize=(9,5))
    # title
    ax.set_title(f'{flt_pct_negative:0.4}% = 0, {flt_pct_positive:0.4}% = 1, (N = {int_total})')
    # frequency bar plot
    ax.bar([0, 1], [int_tot_neg, int_tot_pos])
    # ylabel
    ax.set_ylabel('Frequency')
    # xticks
    ax.set_xticks([0, 1])
    # xtick labels
    ax.set_xticklabels(['0','1'])
    # save
    plt.savefig(f'{str_dirname_output}/{str_filename}', bbox_inches='tight')
    # close
    plt.close()

# df info summary table
def get_df_info_summary_table(dict_df_info_train, dict_df_info_valid, dict_df_info_test, str_dirname_output):
    # create df
    df_info_summary = pd.DataFrame({
        'Data Set': ['Train','Valid','Test'],
        'Rows': [dict_df_info_train['int_nrows'], dict_df_info_valid['int_nrows'], dict_df_info_test['int_nrows']],
        'Columns': [dict_df_info_train['int_ncols'], dict_df_info_valid['int_ncols'], dict_df_info_test['int_ncols']],
        'Total Obs.': [dict_df_info_train['int_obs_total'], dict_df_info_valid['int_obs_total'], dict_df_info_test['int_obs_total']],
        'Min. Date': [dict_df_info_train['date_min'], dict_df_info_valid['date_min'], dict_df_info_test['date_min']],
        'Max. Date': [dict_df_info_train['date_max'], dict_df_info_valid['date_max'], dict_df_info_test['date_max']],
        'Prop. NaN': [dict_df_info_train['flt_propna'], dict_df_info_valid['flt_propna'], dict_df_info_test['flt_propna']],
        'Target Mean': [dict_df_info_train['flt_mean_target'], dict_df_info_valid['flt_mean_target'], dict_df_info_test['flt_mean_target']],
    })
    # save
    df_info_summary.to_csv(f'{str_dirname_output}/df_info_summary.csv', index=False)
    # return df_info_summary
    return df_info_summary

# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

Writing functions.py


### Write ```script.py``` to local drive

In [5]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import functions as func
import json

# constants
str_project = '20231010-gen-xii'
str_dirname_output = './output'

str_id = 'uniqueid'
str_datecol = 'applicationdate__app'
str_target = 'target'
str_model = '02_pricing_pd'

list_cols_id = [
    str_id,
    str_datecol,
    str_target,
]

# output
try:
    os.mkdir(str_dirname_output)
except:
    pass

# iterate through data sets
list_df = [
    'train',
    'valid',
    'test',
]
for str_df in list_df:
    # print
    print(f'Data set: {str_df}')
    # import data
    str_filename = f'df_{str_df}_noleaks_pre.gzip'
    str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename}'
    # read from s3
    df = pd.read_parquet(str_uri)
    # replace
    df.replace(['NaN','nan',''], np.nan, inplace=True)
    
    # get df info
    str_filename = f'dict_df_info_{str_df}.json'
    func.get_df_info(
        df=df, 
        str_datecol=str_datecol, 
        str_dirname_output=str_dirname_output, 
        str_filename=str_filename,
    )
    
    # get descriptives
    str_filename = f'df_descriptives_{str_df}.csv'
    list_cols = [col for col in df.columns if col not in list_cols_id]
    df_tmp = func.get_descriptives_by_column(
        df=df[list_cols], 
        str_dirname_output=str_dirname_output, 
        str_filename=str_filename,
    )
    del df_tmp
    
    # get descriptives of ID cols
    str_filename = f'df_descriptives_id_{str_df}.csv'
    df_tmp = func.get_descriptives_by_column(
        df=df[list_cols_id], 
        str_dirname_output=str_dirname_output, 
        str_filename=str_filename,
    )
    del df_tmp
    
    # plot proportion NaN
    str_filename = f'plt_prop_nan_{str_df}.png'
    func.plot_proportion_nan(
        df=df,
        str_dirname_output=str_dirname_output,
        str_filename=str_filename,
    )
    
    # plot data type frequency
    str_filename = f'plt_dtype_{str_df}.png'
    func.plot_data_type_frequency(
        df=df,
        str_dirname_output=str_dirname_output,
        str_filename=str_filename,
    )
    
    # plot target
    str_filename = f'plt_target_{str_df}.png'
    func.plot_target(
        ser_target=df[str_target],
        str_dirname_output=str_dirname_output,
        str_filename=str_filename,
    )
    print('')

# create table
func.get_df_info_summary_table(
    dict_df_info_train=json.load(open(f'{str_dirname_output}/dict_df_info_train.json')), 
    dict_df_info_valid=json.load(open(f'{str_dirname_output}/dict_df_info_valid.json')), 
    dict_df_info_test=json.load(open(f'{str_dirname_output}/dict_df_info_test.json')),
    str_dirname_output=str_dirname_output,
)

# upload to s3
list_files = os.listdir(str_dirname_output)
for str_filename in list_files:
    str_local_path = f'{str_dirname_output}/{str_filename}'
    str_bucket_path = f'{str_model}/02_model/00_preprocessing/04_eda/{str_filename}'
    func.upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )

Writing script.py


### Build and push to ECR

In [6]:
%%sh

# name the image
image=genxii-pd-eda

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  43.01kB
Step 1/8 : FROM python:3.9
 ---> 7ef94ac333fa
Step 2/8 : RUN apt-get update
 ---> Using cache
 ---> 529128453704
Step 3/8 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 8f56313abc92
Step 4/8 : COPY requirements.txt .
 ---> baaa33e8280c
Step 5/8 : RUN pip install -r requirements.txt
 ---> Running in 0eeeab10bdd7
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.5/159.5 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 35.9 MB/s eta 0:0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.0/305.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 754.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.5/505.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 877.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 730.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pd-eda' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-eda]
ff01422e93ab: Preparing
e89a091f33e7: Preparing
4f3908421a33: Preparing
6e65b0527350: Preparing
7f7a9ee63288: Preparing
781f058a9424: Preparing
78ecb2a2f011: Preparing
84062ebc4cf5: Preparing
2180aea5f54b: Preparing
86388e04a96b: Preparing
893507f6057f: Preparing
2353f7120e0e: Preparing
51a9318e6edf: Preparing
c5bb35826823: Preparing
84062ebc4cf5: Waiting
2180aea5f54b: Waiting
86388e04a96b: Waiting
893507f6057f: Waiting
2353f7120e0e: Waiting
51a9318e6edf: Waiting
c5bb35826823: Waiting
6e65b0527350: Waiting
7f7a9ee63288: Waiting
781f058a9424: Waiting
78ecb2a2f011: Waiting
ff01422e93ab: Pushed
e89a091f33e7: Pushed
6e65b0527350: Pushed
78ecb2a2f011: Pushed
7f7a9ee63288: Pushed
84062ebc4cf5: Pushed
781f058a9424: Pushed
86388e04a96b: Pushed
2180aea5f54b: Pushed
51a9318e6edf: Pushed
c5bb35826823: Pushed
2353f7120e0e: Pushed
4f3908421a33: Pushed
893507f6057f: Pushed
latest: digest: sha256:fa8978e4cfae62

### Clean-up

In [7]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py','functions.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass